# Model Quality Comparison

In [12]:
from pathlib import Path
import ast

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize


# Paths

project_directory = Path(
    r"C:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents"
    r"\IMDB_WEEK1_SIX\movie-embeddings-project"
)

dataset_path = (
    project_directory
    / "data"
    / "processed"
    / "movies_cleaned.csv"
)

embeddings_directory = (
    project_directory
    / "artifacts"
    / "embeddings"
)

metadata_directory = (
    project_directory
    / "artifacts"
    / "metadata"
)

metrics_directory = (
    project_directory
    / "artifacts"
    / "metrics"
)

genres_path = metadata_directory / "movie_genres.csv"

metrics_directory.mkdir(parents=True, exist_ok=True)


# Load movies

movies_df = pd.read_csv(dataset_path).reset_index(drop=True)
movies_df["movie_id"] = movies_df["movie_id"].astype(int)


# Load embeddings

embedding_paths = {
    "minilm": embeddings_directory / "minilm_embeddings.npy",
    "mpnet": embeddings_directory / "mpnet_embeddings.npy",
    "gte_modernbert": (
        embeddings_directory / "gte_modernbert_embeddings.npy"
    ),
}

embeddings = {}

for model_name, embedding_path in embedding_paths.items():
    model_embeddings = np.load(
        embedding_path
    ).astype(np.float32)

    embeddings[model_name] = normalize(
        model_embeddings,
        norm="l2",
        axis=1,
    )

    print(
        f"{model_name}: "
        f"shape={embeddings[model_name].shape}, "
        f"dtype={embeddings[model_name].dtype}"
    )


# Load genres

genres_df = pd.read_csv(genres_path)
genres_df["movie_id"] = genres_df["movie_id"].astype(int)

genres_df["genre_set"] = genres_df["genres"].apply(
    lambda value: (
        set(ast.literal_eval(value))
        if pd.notna(value) and value != "[]"
        else set()
    )
)

genre_lookup = (
    genres_df
    .set_index("movie_id")["genre_set"]
    .to_dict()
)

movies_df["genres"] = movies_df["movie_id"].map(
    genre_lookup
)

print(f"\nBroj filmova: {len(movies_df)}")
print(f"Broj modela: {len(embeddings)}")
print(
    f"Filmovi sa žanrovima: "
    f"{movies_df['genres'].apply(bool).sum()}"
)

minilm: shape=(9967, 384), dtype=float32
mpnet: shape=(9967, 768), dtype=float32
gte_modernbert: shape=(9967, 768), dtype=float32

Broj filmova: 9967
Broj modela: 3
Filmovi sa žanrovima: 9965


In [13]:
query_titles = [
    "The Dark Knight",
    "Inception",
    "Toy Story",
    "The Godfather",
    "The Matrix",
    "Titanic",
    "Alien",
    "The Exorcist",
    "Saving Private Ryan",
    "The Notebook",
    "Finding Nemo",
    "Se7en",
    "The Silence of the Lambs",
    "The Lord of the Rings: The Fellowship of the Ring",
    "The Truman Show",
    "Black Swan",
    "Mad Max: Fury Road",
    "The Social Network",
    "The Grand Budapest Hotel",
]


def get_similar_movies(
    movies: pd.DataFrame,
    model_embeddings: np.ndarray,
    query_movie_id: int,
    top_k: int = 10,
) -> pd.DataFrame:
    query_position = movies.index[
        movies["movie_id"] == query_movie_id
    ][0]

    similarities = cosine_similarity(
        model_embeddings[query_position].reshape(1, -1),
        model_embeddings,
    ).ravel()

    similarities[query_position] = -np.inf

    top_positions = np.argsort(
        similarities
    )[::-1][:top_k]

    results = movies.iloc[
        top_positions
    ][
        [
            "movie_id",
            "title",
        ]
    ].copy()

    results.insert(
        0,
        "rank",
        range(1, len(results) + 1),
    )

    results["cosine_similarity"] = similarities[
        top_positions
    ]

    return results.reset_index(drop=True)


selected_query_movies = []

for title in query_titles:
    matches = movies_df[
        movies_df["title"].str.casefold()
        == title.casefold()
    ]

    if len(matches) == 1:
        selected_query_movies.append(
            {
                "query_movie_id": int(
                    matches.iloc[0]["movie_id"]
                ),
                "query_title": title,
            }
        )
    else:
        print(
            f"Preskočen '{title}', "
            f"broj pronađenih verzija: {len(matches)}"
        )

print(
    f"Broj izabranih query filmova: "
    f"{len(selected_query_movies)}"
)


Broj izabranih query filmova: 19


In [14]:
top_k = 10
comparison_rows = []

for query_movie in selected_query_movies:
    query_movie_id = query_movie["query_movie_id"]
    query_title = query_movie["query_title"]
    query_genres = genre_lookup.get(query_movie_id, set())

    for model_name, model_embeddings in embeddings.items():
        recommendations = get_similar_movies(
            movies=movies_df,
            model_embeddings=model_embeddings,
            query_movie_id=query_movie_id,
            top_k=top_k,
        )

        for _, row in recommendations.iterrows():
            recommended_movie_id = int(row["movie_id"])
            recommended_genres = genre_lookup.get(
                recommended_movie_id,
                set(),
            )

            shared_genres = query_genres & recommended_genres
            all_genres = query_genres | recommended_genres

            comparison_rows.append(
                {
                    "query_movie_id": query_movie_id,
                    "query_title": query_title,
                    "model": model_name,
                    "rank": int(row["rank"]),
                    "recommended_movie_id": recommended_movie_id,
                    "recommended_title": row["title"],
                    "cosine_similarity": float(
                        row["cosine_similarity"]
                    ),
                    "query_genres": ", ".join(
                        sorted(query_genres)
                    ),
                    "recommended_genres": ", ".join(
                        sorted(recommended_genres)
                    ),
                    "shared_genres": ", ".join(
                        sorted(shared_genres)
                    ),
                    "genre_overlap": int(bool(shared_genres)),
                    "genre_jaccard": (
                        len(shared_genres) / len(all_genres)
                        if all_genres
                        else 0.0
                    ),
                }
            )

comparison_df = pd.DataFrame(comparison_rows)

comparison_path = (
    metrics_directory
    / "cosine_similarity_with_genres.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False,
)

print(f"Broj rezultata: {len(comparison_df)}")
print(f"Sačuvano u: {comparison_path}")

display(comparison_df.head(20))

Broj rezultata: 570
Sačuvano u: C:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\IMDB_WEEK1_SIX\movie-embeddings-project\artifacts\metrics\cosine_similarity_with_genres.csv


,query_movie_id,query_title,model,rank,recommended_movie_id,recommended_title,cosine_similarity,query_genres,recommended_genres,shared_genres,genre_overlap,genre_jaccard
0,155,The Dark Knight,minilm,1,736073,"Batman: The Long Halloween, Part One",0.764030,"Action, Crime, Thriller","Action, Animation, Crime, Mystery","Action, Crime",1,0.400000
1,155,The Dark Knight,minilm,2,736074,"Batman: The Long Halloween, Part Two",0.725439,"Action, Crime, Thriller","Action, Animation, Crime, Mystery","Action, Crime",1,0.400000
2,155,The Dark Knight,minilm,3,268,Batman,0.707406,"Action, Crime, Thriller","Action, Crime, Fantasy","Action, Crime",1,0.500000
3,155,The Dark Knight,minilm,4,49026,The Dark Knight Rises,0.701888,"Action, Crime, Thriller","Action, Crime, Drama, Thriller","Action, Crime, Thriller",1,0.750000
4,155,The Dark Knight,minilm,5,382322,Batman: The Killing Joke,0.648050,"Action, Crime, Thriller","Animation, Crime, Drama",Crime,1,0.200000
5,155,The Dark Knight,minilm,6,471474,Batman: Gotham by Gaslight,0.647677,"Action, Crime, Thriller","Action, Animation, Science Fiction, Thriller","Action, Thriller",1,0.400000
6,155,The Dark Knight,minilm,7,209112,Batman v Superman: Dawn of Justice,0.641332,"Action, Crime, Thriller","Action, Adventure, Fantasy",Action,1,0.200000
7,155,The Dark Knight,minilm,8,414906,The Batman,0.633064,"Action, Crime, Thriller","Crime, Mystery, Thriller","Crime, Thriller",1,0.500000
8,155,The Dark Knight,minilm,9,21683,Batman: Mystery of the Batwoman,0.622899,"Action, Crime, Thriller","Action, Adventure, Animation, Crime, Drama, Fa...","Action, Crime",1,0.285714
9,155,The Dark Knight,minilm,10,272,Batman Begins,0.614793,"Action, Crime, Thriller","Action, Crime, Drama","Action, Crime",1,0.500000


In [ ]:
evaluation_rows = []

for top_k in [5, 10]:
    top_results = comparison_df[
        comparison_df["rank"] <= top_k
    ]

    per_query = (
        top_results
        .groupby(
            [
                "model",
                "query_movie_id",
                "query_title",
            ],
            as_index=False,
        )
        .agg(
            genre_overlap_at_k=(
                "genre_overlap",
                "mean",
            ),
            genre_jaccard_at_k=(
                "genre_jaccard",
                "mean",
            ),
        )
    )

    per_query["top_k"] = top_k
    evaluation_rows.append(per_query)

evaluation_df = pd.concat(
    evaluation_rows,
    ignore_index=True,
)

model_quality_summary = (
    evaluation_df
    .groupby(
        [
            "model",
            "top_k",
        ],
        as_index=False,
    )
    .agg(
        genre_overlap_mean=(
            "genre_overlap_at_k",
            "mean",
        ),
        genre_jaccard_mean=(
            "genre_jaccard_at_k",
            "mean",
        ),
    )
    .sort_values(
        [
            "top_k",
            "genre_jaccard_mean",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

display(model_quality_summary)

,model,top_k,genre_overlap_mean,genre_jaccard_mean
0,gte_modernbert,5,0.810526,0.463471
4,mpnet,5,0.831579,0.434261
2,minilm,5,0.778947,0.405138
1,gte_modernbert,10,0.847368,0.430157
5,mpnet,10,0.831579,0.424236
3,minilm,10,0.805263,0.389160


GTE najbolji model prema Jaccard@5 i @10 i overlap@10. na osnovu ovih filmova se ne vidi jasno ali kada bude najbolji razlika je znatno veca.

In [ ]:
evaluation_df.to_csv(
    metrics_directory
    / "genre_evaluation_per_query.csv",
    index=False,
)

model_quality_summary.to_csv(
    metrics_directory
    / "genre_evaluation_summary.csv",
    index=False,
)

print("Rezultati su sačuvani!")

Rezultati su sačuvani.
